# Singular Value Decomposition — Wholesale Customers Data Set

We treat each wholesale client as a row and the six annual
spending channels as columns. After **centering** each
column, truncated SVD gives a low-rank factorization that
preserves as much Frobenius-norm energy as possible — the
same Eckart–Young principle as PCA, without an explicit PCA
object.

$$X \approx U_k \Sigma_k V_k^\top$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rice_ml.processing.pre_processing import find_data_file

np.random.seed(0)
plt.rcParams["figure.figsize"] = (7, 4.5)

from rice_ml.unsupervised_learning.decomposition import SVD

df = pd.read_csv(find_data_file("Wholesale_customers_data.csv"))
spend_cols = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
X = df[spend_cols].to_numpy(dtype=float)
Xc = X - X.mean(axis=0, keepdims=True)

In [ ]:
k = 2
U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
approx2 = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
im0 = axes[0].imshow(Xc.T, aspect="auto", cmap="magma")
axes[0].set_title("centered X (features × clients)")
axes[0].set_ylabel("feature index")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(approx2.T, aspect="auto", cmap="magma")
axes[1].set_title("rank-2 SVD approximation")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow((Xc - approx2).T, aspect="auto", cmap="coolwarm")
axes[2].set_title("residual X - X̂")
plt.colorbar(im2, ax=axes[2], fraction=0.046)
plt.tight_layout()
plt.show()

In [ ]:
svd = SVD(n_components=6).fit(Xc)
Z = svd.transform(Xc)
print(f"latent shape: {Z.shape}")
print(f"singular values: {svd.singular_values_.round(2)}")

cum = np.cumsum(svd.explained_variance_ratio_)
fig, ax = plt.subplots()
ax.plot(np.arange(1, len(cum) + 1), cum, marker="o")
ax.set_xlabel("# of components")
ax.set_ylabel("cumulative explained variance")
ax.set_title("Truncated SVD (centered wholesale spending)")
plt.show()

## Takeaways

- Leading singular triple captures the strongest
  co-spending pattern across clients.
- Centering aligns this demo with PCA-style variance; raw
  SVD without centering emphasizes the overall magnitude
  direction instead.
- Latent rows `Z` can be fed to clustering or visualization
  instead of the raw six channels.